# Phase 6 — Dual-Path Stage 1 + Stage 2 Evaluation

Evaluate the Phase 6 trained Dual-Path Stage 1 (Path A causal + Path B UNet + FusionGate)
combined with the Stage 2 EDM-Karras diffusion decoder (loaded from prior causal training)
on the validation split. Compute the full metric battery:

- Deterministic: RMSE_final, Pearson, F1@p95, F1@p99
- Probabilistic: CRPS_log1p, spread, spread/RMSE ratio
- Climate (via aligned_eval): CDD, Rx1day, R10day biases, PSD distance

Inputs
------
- `epoch_best_dualpath.pth` : Phase 6 output (encoder + RCN + reg_head + dual_path)
- `epoch_last.pth`          : prior causal training (provides Stage 2 diffusion weights)
- `sigma_data` = 0.193      : recalibrated by Phase 6

Output
------
- `phase6_evaluation/phase6_eval_results.json`


In [ ]:
# === Cell 1 : Bootstrap Colab ===
import subprocess, shlex, os, sys
from pathlib import Path

REPO_DIR   = Path('/content/climate_data')
GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
GIT_BRANCH = 'four-node-causal'

if not (REPO_DIR / '.git').exists():
    subprocess.run(shlex.split(
        f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
else:
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
    subprocess.run(shlex.split(
        f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)

os.chdir(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))

try:
    import torch_geometric; import cftime; import h5netcdf; import xbatcher; import diffusers
    from omegaconf import OmegaConf
except ImportError:
    EXTRA_DEPS = [
        'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
        'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
        'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
        'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
    ]
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + EXTRA_DEPS, check=True)

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ModuleNotFoundError:
    print('[bootstrap] not on Colab — Drive mount skipped')

print(f'[bootstrap] cwd={os.getcwd()}  branch={GIT_BRANCH}')


In [ ]:
# === Cell 2 : Imports + Constantes Phase 6 Eval ===
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
from pathlib import Path
from omegaconf import OmegaConf

from st_cdgm.models.dual_path_stage1 import DualPathPredictor, PathBCNN, FusionGate
from st_cdgm.training.stage1_paths import (
    predict_mu_hr_dualpath,
    batch_lr_grid_last,
    _as_batched_hr,
)

# --- Chemins Drive ---
DRIVE_ROOT     = Path('/content/drive/MyDrive/climate_data')
ORACLE_9N      = DRIVE_ROOT / 'oracle_9node' / 'seed_42'

# Path A + dual_path + new mu_HR (sortie Phase 6)
CKPT_DUALPATH  = ORACLE_9N / 'epoch_best_dualpath.pth'

# Stage 2 (diffusion) : provient du training Path C+ Option C 9-node original.
# Dans le layout reel des notebooks 9-node, epoch_last.pth est directement dans
# oracle_9node/seed_42/ (pas dans un sous-dossier ckpt_causal/).
CKPT_STAGE2    = ORACLE_9N / 'epoch_last.pth'

# Recalibrated sigma_data from Phase 6 (was 0.18856 in Phase 5)
SIGMA_DATA_NEW = 0.193

# Eval params (match Phase 5 evaluation)
K_SAMPLES      = 4
N_STEPS        = 32
N_BATCHES_EVAL = 90  # match Phase 5 in-distribution eval

# Output dir
OUT_DIR        = ORACLE_9N / 'phase6_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Validations ---
assert CKPT_DUALPATH.exists(), f'Phase 6 dual-path checkpoint introuvable : {CKPT_DUALPATH}'
assert CKPT_STAGE2.exists(),   f'Stage 2 checkpoint introuvable : {CKPT_STAGE2}'

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'[Cell 2] DEVICE         = {DEVICE}')
print(f'[Cell 2] CKPT_DUALPATH  = {CKPT_DUALPATH}')
print(f'[Cell 2] CKPT_STAGE2    = {CKPT_STAGE2}')
print(f'[Cell 2] SIGMA_DATA_NEW = {SIGMA_DATA_NEW}')
print(f'[Cell 2] K_SAMPLES      = {K_SAMPLES}  N_STEPS = {N_STEPS}  N_BATCHES_EVAL = {N_BATCHES_EVAL}')
print(f'[Cell 2] OUT_DIR        = {OUT_DIR}')


In [ ]:
# === Cell 3 : Config + Pipeline + Dataloaders (9-node) ===
from torch.utils.data import DataLoader as _DataLoader, IterableDataset
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from path_c_plus.scripts.option_c_helpers import PATHCPLUS_HYPERPARAM_OVERRIDES
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

# --- Config ---
CONFIG = OmegaConf.load('config/training_config.yaml')
_corrdiff = OmegaConf.load('config/training_config_corrdiff_normal.yaml')
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

EXTENDED_9NODE = True
GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Forcer batch_size=1 pour la compatibilite single-sample (IterableDataset)
CONFIG.training.batch_size  = 1
CONFIG.training.use_amp     = GPU_PROFILE['use_amp']
CONFIG.training.num_workers = GPU_PROFILE['num_workers']

ts_cfg = CONFIG.two_stage
ts_cfg.stage1['lambda_dag_prior'] = PATHCPLUS_HYPERPARAM_OVERRIDES['lambda_dag_prior']
ts_cfg.stage1['g_phys_alpha']     = PATHCPLUS_HYPERPARAM_OVERRIDES['g_phys_alpha']

# --- Ajout metapaths 9-node ---
OmegaConf.set_struct(CONFIG, False)
_existing_mp = {m.name for m in CONFIG.encoder.metapaths}
for _m in [
    {'name': 'Q850', 'src': 'Q850', 'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
    {'name': 'W500', 'src': 'W500', 'relation': 'causes', 'target': 'GP500', 'pool': 'mean'},
    {'name': 'IVT',  'src': 'IVT',  'relation': 'causes', 'target': 'GP850', 'pool': 'mean'},
]:
    if _m['name'] not in _existing_mp:
        CONFIG.encoder.metapaths.append(OmegaConf.create(_m))
print(f'[Cell 3] metapaths -> {[m.name for m in CONFIG.encoder.metapaths]}')

# --- Dates ---
K9_DATES = {
    'train':   ['1980-01-01', '2009-12-31'],
    'val':     ['2010-01-01', '2011-12-31'],
    'test':    ['2012-01-01', '2013-12-31'],
    'holdout': ['2014-01-01', '2014-12-31'],
}

_ON_COLAB  = 'google.colab' in sys.modules or Path('/content').exists()
DATA_ROOT  = Path('/content/drive/MyDrive/climate_data/data') if _ON_COLAB else Path('data/raw')
LR_PATH    = str(DATA_ROOT / 'train' / 'predictor_ACCESS-CM2_hist.nc')
HR_PATH    = str(DATA_ROOT / 'train' / 'pr_ACCESS-CM2_hist.nc')
_static_p  = DATA_ROOT / 'static_predictors' / 'ERA5_eval_ccam_12km.198110_NZ_Invariant.nc'
_mean_p    = DATA_ROOT / 'train' / 'means_ACCESS-CM2.nc'
_std_p     = DATA_ROOT / 'train' / 'stds_ACCESS-CM2.nc'
STATIC_PATH = str(_static_p) if _static_p.exists() else None
MEAN_PATH   = str(_mean_p)   if _mean_p.exists()   else None
STD_PATH    = str(_std_p)    if _std_p.exists()    else None

SEQ_LEN             = int(CONFIG.data.seq_len)
BASELINE_STRATEGY   = str(CONFIG.data.baseline_strategy)
BASELINE_FACTOR     = int(CONFIG.data.baseline_factor)
NORMALIZE           = bool(CONFIG.data.normalize)
PRECIPITATION_DELTA = float(CONFIG.data.precipitation_delta)
NAN_FILL_STRATEGY   = str(CONFIG.data.nan_fill_strategy)
_default_lr = ['q_500', 'q_850', 'u_500', 'u_850', 'v_500', 'v_850', 't_500', 't_850']
LR_VARIABLES  = list(CONFIG.data.lr_variables)  if CONFIG.data.get('lr_variables')  else _default_lr
HR_VARIABLES  = list(CONFIG.data.hr_variables)  if CONFIG.data.get('hr_variables')  else ['pr']
STATIC_VARIABLES = list(CONFIG.data.static_variables) if CONFIG.data.get('static_variables') else []

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=SEQ_LEN, baseline_strategy=BASELINE_STRATEGY,
    baseline_factor=BASELINE_FACTOR, normalize=NORMALIZE,
    nan_fill_strategy=NAN_FILL_STRATEGY,
    precipitation_delta=PRECIPITATION_DELTA,
    lr_variables=LR_VARIABLES, hr_variables=HR_VARIABLES,
    static_variables=STATIC_VARIABLES,
    means_path=MEAN_PATH, stds_path=STD_PATH,
    train_start_date=K9_DATES['train'][0], train_end_date=K9_DATES['train'][1],
    val_start_date=K9_DATES['val'][0],     val_end_date=K9_DATES['val'][1],
    test_start_date=K9_DATES['test'][0],   test_end_date=K9_DATES['test'][1],
    temporal_holdout_start_date=K9_DATES['holdout'][0],
    temporal_holdout_end_date=K9_DATES['holdout'][1],
)
print('[Cell 3] Pipeline ready')

train_dataset = pipeline.build_sequence_dataset(split='train', seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)
val_dataset   = pipeline.build_sequence_dataset(split='val',   seq_len=SEQ_LEN,
                                                 stride=int(CONFIG.data.stride), as_torch=True)

BATCH_SIZE  = 1
NUM_WORKERS = int(CONFIG.training.num_workers)
PIN_MEMORY  = bool(torch.cuda.is_available())
_loader_kw  = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                   pin_memory=PIN_MEMORY, collate_fn=lambda x: x)
if NUM_WORKERS > 0:
    _loader_kw.update(persistent_workers=True, prefetch_factor=2)

train_dataloader = _DataLoader(train_dataset,
                                shuffle=not isinstance(train_dataset, IterableDataset),
                                **_loader_kw)
val_dataloader   = _DataLoader(val_dataset, shuffle=False, **_loader_kw)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder  = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=CONFIG.graph.include_mid_layer,
    extended_9node=EXTENDED_9NODE,
)
print(f'[Cell 3] Builder  lr_shape={lr_shape}  hr_shape={hr_shape}  dyn={builder.dynamic_node_types}')

# --- convert_sample_to_batch (identique training, avec lr_grid pour batch_lr_grid_last) ---
_LR_VARS = list(CONFIG.data.lr_variables)
_VI = {v: i for i, v in enumerate(_LR_VARS)}
_Q_IDX = [_VI[v] for v in ('q_850', 'q_500', 'q_250') if v in _VI]
_W_IDX = [_VI[v] for v in ('w_850', 'w_500', 'w_250') if v in _VI]
_IVT_LEVELS = [lev for lev in ('850', '500', '250')
               if f'q_{lev}' in _VI and f'u_{lev}' in _VI and f'v_{lev}' in _VI]

def _compute_ivt_nodes(lr0):
    acc = None
    for lev in _IVT_LEVELS:
        q = lr0[:, [_VI[f'q_{lev}']]]
        u = lr0[:, [_VI[f'u_{lev}']]]
        v = lr0[:, [_VI[f'v_{lev}']]]
        term = q * torch.sqrt(u * u + v * v + 1e-12)
        acc = term if acc is None else acc + term
    if acc is None:
        acc = lr0[:, 0:1] * 0.0
    return acc / (len(_IVT_LEVELS) + 1e-8)

def _ensure_2d(t):
    return t.unsqueeze(-1) if t.dim() == 1 else t

def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample['lr']
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    lr0 = lr_nodes_steps[0]
    if EXTENDED_9NODE:
        _ivt = _compute_ivt_nodes(lr0)
        dynamic_features = {}
        for nt in builder.dynamic_node_types:
            if nt == 'Q850':   dynamic_features[nt] = _ensure_2d(lr0[:, _Q_IDX] if _Q_IDX else lr0)
            elif nt == 'W500': dynamic_features[nt] = _ensure_2d(lr0[:, _W_IDX] if _W_IDX else lr0)
            elif nt == 'IVT':  dynamic_features[nt] = _ensure_2d(_ivt)
            else:              dynamic_features[nt] = _ensure_2d(lr0)
    else:
        dynamic_features = {nt: _ensure_2d(lr0) for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {
        'lr':       lr_tensor,
        'lr_grid':  lr_seq,
        'residual': sample['residual'],
        'baseline': sample.get('baseline'),
        'hetero':   hetero,
        'time':     sample.get('time'),
    }

def iterate_batches(dataloader, builder, device):
    for batch_list in dataloader:
        if not isinstance(batch_list, list):
            batch_list = [batch_list]
        yield [convert_sample_to_batch(s, builder, device) for s in batch_list]

_probe = next(iter(train_dataset))
C_LR = _probe['lr'].shape[1]
_n_train = len(train_dataset) if hasattr(train_dataset, '__len__') else '?'
_n_val   = len(val_dataset)   if hasattr(val_dataset,   '__len__') else '?'
print(f'[Cell 3] C_LR={C_LR}  lr_shape={lr_shape}  hr_shape={hr_shape}')
print(f'[Cell 3] train={_n_train} samples  val={_n_val} samples')

# Constantes pour HR shape utilises plus tard pour DualPathPredictor
H_HR, W_HR = int(hr_shape[0]), int(hr_shape[1])
print(f'[Cell 3] HR shape = ({H_HR}, {W_HR})')


In [ ]:
# === Cell 4 : Charger Stage 2 (diffusion decoder) ===
# On reutilise le pattern build_stack du notebook v5_evaluation mais on n'extrait
# QUE le diffusion_decoder ; Stage 1 vient du checkpoint Phase 6 (Cell 5).
from st_cdgm.models import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from omegaconf import OmegaConf as _OC

print(f'[Cell 4] Chargement Stage 2 : {CKPT_STAGE2}')
ck_s2 = torch.load(CKPT_STAGE2, map_location=DEVICE, weights_only=False)
print(f'[Cell 4] Cles disponibles : {sorted(ck_s2.keys())[:12]} ...')

# --- Build diffusion module (idem build_stack du v5_eval notebook) ---
edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get('edm', {}))

_unet_kwargs = _OC.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ('down_block_types', 'up_block_types'):
    if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
        _unet_kwargs[_k] = tuple(_unet_kwargs[_k])

# hr_channels via probe (= 1 pour la precipitation)
_probe_sample = next(iter(val_dataset))
hr_channels = int(_probe_sample['residual'].shape[1])

diffusion_decoder = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=_unet_kwargs,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get('use_gradient_checkpointing', False)),
    conv_padding_mode=str(CONFIG.diffusion.get('conv_padding_mode', 'zeros')),
    anti_checkerboard=bool(CONFIG.diffusion.get('anti_checkerboard', False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)

# --- Strip prefixes torch.compile / DDP ---
def _strip_prefixes(sd):
    if sd is None:
        return None
    prefixes = ['_orig_mod.', 'module.']
    out = {}
    for k, v in sd.items():
        nk = k
        for p in prefixes:
            if nk.startswith(p):
                nk = nk[len(p):]
        out[nk] = v
    return out

_diff_sd = _strip_prefixes(ck_s2.get('diffusion_state_dict'))
if _diff_sd is None:
    raise RuntimeError(
        f'diffusion_state_dict absent de {CKPT_STAGE2}. '
        f'Le checkpoint Stage 2 doit etre le epoch_last.pth du training Path C+ '
        f'Option C 9-node.'
    )
_missing, _unexpected = diffusion_decoder.load_state_dict(_diff_sd, strict=False)
if _missing:
    print(f'[Cell 4] missing keys : {len(_missing)} (premieres : {_missing[:3]})')
if _unexpected:
    print(f'[Cell 4] unexpected keys : {len(_unexpected)} (premieres : {_unexpected[:3]})')

# --- Override sigma_data avec la valeur recalibree Phase 6 ---
_old_sigma = diffusion_decoder.edm_config.sigma_data
diffusion_decoder.edm_config.sigma_data = SIGMA_DATA_NEW
print(f'[Cell 4] sigma_data : {_old_sigma:.5f} -> {SIGMA_DATA_NEW} (override Phase 6)')

# --- Freeze + eval ---
for p in diffusion_decoder.parameters():
    p.requires_grad_(False)
diffusion_decoder.eval()

_n_diff = sum(p.numel() for p in diffusion_decoder.parameters())
print(f'[Cell 4] Stage 2 loaded from {CKPT_STAGE2}, sigma_data={SIGMA_DATA_NEW}')
print(f'[Cell 4] diffusion params = {_n_diff:,}')


In [ ]:
# === Cell 5 : Charger Phase 6 Dual-Path Stage 1 ===
# Reproduit le pattern training Cell 4 + Cell 5 (build encoder, RCN, head,
# dual_path) en mode eval/inference seulement.
import re
from st_cdgm.models.intelligible_encoder import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
)
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.regression_head import GraphToGridDecoder

def _parse_encoder_metapaths_from_ckpt(enc_sd):
    """Infer ordered (name, src, rel, target) list from checkpoint encoder keys.
    Format: metapath_convs.{name}__{src}__{rel}__{target}.{param}
    """
    seen = {}
    order = []
    for k in enc_sd:
        if not k.startswith('metapath_convs.'):
            continue
        rest = k[len('metapath_convs.'):]
        parts = rest.split('__')
        if len(parts) < 4:
            continue
        name = parts[0]
        src  = parts[1]
        rel  = parts[2]
        tgt  = parts[3].split('.')[0]
        if name not in seen:
            seen[name] = (src, rel, tgt)
            order.append(name)
    return [(n,) + seen[n] for n in order]

def _build_encoder_from_ckpt(enc_sd, CONFIG, device):
    """Build encoder whose metapath configs match exactly the checkpoint keys."""
    parsed = _parse_encoder_metapaths_from_ckpt(enc_sd)
    print(f'  Metapaths detectes : {[t[0] for t in parsed]}')
    cfgs = [
        IntelligibleVariableConfig(name=name, meta_path=(src, rel, tgt), pool='mean')
        for name, src, rel, tgt in parsed
    ]
    enc = IntelligibleVariableEncoder(
        configs=cfgs,
        hidden_dim=int(CONFIG.encoder.hidden_dim),
        conditioning_dim=int(CONFIG.encoder.conditioning_dim),
    ).to(device)
    return enc, len(cfgs)

# --- Charger checkpoint ---
print(f'[Cell 5] Chargement : {CKPT_DUALPATH}')
ck_dp = torch.load(CKPT_DUALPATH, map_location=DEVICE, weights_only=False)
print(f'[Cell 5] Cles dans checkpoint : {sorted(ck_dp.keys())[:12]} ...')

def _clean_sd(sd):
    if sd is None:
        return None
    if any('_orig_mod' in k for k in sd):
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    return sd

_enc_sd = _clean_sd(ck_dp.get('encoder_state_dict', {}))

# --- Build encoder (taille auto-detectee depuis ckpt) ---
torch.manual_seed(SEED); np.random.seed(SEED)
print('[Cell 5] Inference structure encoder depuis checkpoint...')
encoder, num_vars = _build_encoder_from_ckpt(_enc_sd, CONFIG, DEVICE)

# --- RCN driver dim via probe ---
_probe_b = next(iter(val_dataset))
_lr_nodes = builder.lr_grid_to_nodes(_probe_b['lr'][0])
RCN_DRIVER_DIM = _lr_nodes.shape[-1]

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=int(CONFIG.rcn.hidden_dim),
    driver_dim=RCN_DRIVER_DIM,
    reconstruction_dim=RCN_DRIVER_DIM,
    dropout=float(CONFIG.rcn.dropout),
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get('detach_interval'))

rh_cfg = CONFIG.two_stage.regression_head
regression_head = GraphToGridDecoder(
    d_model=int(rh_cfg.d_model),
    hr_h=H_HR, hr_w=W_HR,
    intermediate_h=int(rh_cfg.intermediate_h),
    intermediate_w=int(rh_cfg.intermediate_w),
    n_heads=int(rh_cfg.n_heads),
    refine_channels=int(rh_cfg.refine_channels),
    output_channels=1,
).to(DEVICE)

# --- DualPath (architecture identique au training Phase 6) ---
PATH_B_KIND          = 'unet'
PATH_B_UNET_CHANNELS = (32, 64, 128)
PATH_B_UNET_LR_SHAPE = (23, 26)
PATH_B_BASE_CH       = 48
GATE_MAX_MEAN        = 0.40

dual_path = DualPathPredictor(
    in_channels=C_LR,
    base_ch=PATH_B_BASE_CH,
    hr_h=H_HR, hr_w=W_HR,
    gate_max_mean=GATE_MAX_MEAN,
    path_b_kind=PATH_B_KIND,
    path_b_unet_channels=PATH_B_UNET_CHANNELS,
    path_b_unet_lr_shape=PATH_B_UNET_LR_SHAPE,
).to(DEVICE)

# --- Charger tous les state_dicts (strict=False pour path_b_bias compat) ---
def _safe_load(module, ck_data, keys, label):
    for key in keys:
        sd = ck_data.get(key)
        if sd is not None:
            sd = _clean_sd(sd)
            try:
                missing, unexpected = module.load_state_dict(sd, strict=False)
                msg = f'  [{label}] charge depuis "{key}"'
                if missing:    msg += f'  | missing={len(missing)}'
                if unexpected: msg += f'  | unexpected={len(unexpected)}'
                print(msg)
                return True
            except Exception as e:
                print(f'  [{label}] ERREUR load avec "{key}" : {type(e).__name__}: {e}')
                continue
    print(f'  [{label}] AUCUNE cle valide dans {keys}')
    return False

_safe_load(encoder,         ck_dp, ['encoder_state_dict'],                            'encoder')
_safe_load(rcn_cell,        ck_dp, ['rcn_cell_state_dict', 'rcn_state_dict'],         'rcn_cell')
_safe_load(regression_head, ck_dp, ['regression_head_state_dict', 'head_state_dict'], 'regression_head')
_safe_load(dual_path,       ck_dp, ['dual_path_state_dict'],                          'dual_path')

# --- Freeze tout en eval ---
for m in (encoder, rcn_cell, regression_head, dual_path):
    for p in m.parameters():
        p.requires_grad_(False)
    m.eval()

# --- Verifier A_dag ---
_rcn_core = rcn_cell
if hasattr(_rcn_core, '_orig_mod'):
    _rcn_core = _rcn_core._orig_mod
if hasattr(_rcn_core, 'A_dag'):
    _A_dag = _rcn_core.A_dag.detach()
    print(f'[Cell 5] A_dag  shape={tuple(_A_dag.shape)}  norm={_A_dag.norm():.4f}  '
          f'asym={(_A_dag - _A_dag.T).abs().mean():.4f}')
else:
    print('[Cell 5] WARN : A_dag absent')

# --- Stats ---
_n_enc = sum(p.numel() for p in encoder.parameters())
_n_rcn = sum(p.numel() for p in rcn_cell.parameters())
_n_rh  = sum(p.numel() for p in regression_head.parameters())
_n_dp  = sum(p.numel() for p in dual_path.parameters())
print(f'[Cell 5] Param counts : enc={_n_enc:,}  rcn={_n_rcn:,}  head={_n_rh:,}  dual_path={_n_dp:,}')
print(f'[Cell 5] path_b_bias  : {float(dual_path.path_b_bias.item()):+.5f}')
print(f'[Cell 5] sigma_data Stage 2 effectif : {diffusion_decoder.edm_config.sigma_data}')


In [ ]:
# === Cell 6 : Inference loop (K samples par batch) ===
import time as _time

@torch.no_grad()
def predict_dualpath(batch, K=K_SAMPLES, n_steps=N_STEPS):
    """Run Path A + Path B + gate + K-sample Stage 2 sampling.

    Returns
    -------
    ensemble : Tensor [K, 1, 1, H, W]   K HR predictions in log1p space
    mu_A     : Tensor [1, 1, H, W]
    mu_B     : Tensor [1, 1, H, W]
    mu_total : Tensor [1, 1, H, W]
    """
    # 1. Stage 1 dual-path : mu_A, mu_B, mu_total
    mu_A, mu_B, mu_total, gate = predict_mu_hr_dualpath(
        batch,
        encoder=encoder, rcn_runner=rcn_runner, regression_head=regression_head,
        dual_path=dual_path,
        builder=builder, device=DEVICE,
    )
    # NaN safety
    mu_total = torch.nan_to_num(mu_total, nan=0.0)

    # 2. Baseline_log (= log1p(baseline LR)) requis par causal_concat
    bl = batch['baseline'][-1].to(DEVICE)
    if bl.dim() == mu_total.dim() - 1:
        bl = bl.unsqueeze(0)
    bl = torch.nan_to_num(bl, nan=0.0)

    # 3. K-sample Stage 2 (EDM Karras, num_steps=N_STEPS)
    samples = []
    for _k in range(K):
        out = diffusion_decoder.sample(
            conditioning=None,
            num_steps=n_steps,
            scheduler_type='edm_karras',
            apply_constraints=False,
            mu_HR=mu_total,
            baseline_log=bl,
        )
        r = out.residual if hasattr(out, 'residual') else out
        # Final HR (log1p space) = baseline + mu_total + residual
        hr_pred = bl + mu_total + r
        samples.append(hr_pred)
    ensemble = torch.stack(samples, dim=0)  # [K, 1, 1, H, W]
    return ensemble, mu_A, mu_B, mu_total

# --- Iterate val_dataloader for N_BATCHES_EVAL batches ---
preds_all    = []   # list of [K, 1, 1, H, W]
truths_all   = []   # list of [1, 1, H, W]
mu_total_all = []   # list of [1, 1, H, W]
times_all    = []   # list of timestamps (for aligned_eval)

t0 = _time.time()
n_done = 0
for bi, batch_list in enumerate(iterate_batches(val_dataloader, builder, DEVICE)):
    if bi >= N_BATCHES_EVAL:
        break
    for micro in batch_list:
        ensemble, mu_A, mu_B, mu_total = predict_dualpath(micro, K=K_SAMPLES, n_steps=N_STEPS)
        target = micro['residual'][-1].to(DEVICE)
        # target in dataset is the residual in log1p space; we need full HR =
        # baseline + residual (log1p). For metric comparison, predictions
        # already contain baseline + mu_total + r so we must compare to
        # baseline + true residual.
        bl_t = micro['baseline'][-1].to(DEVICE)
        if bl_t.dim() == target.dim() - 1:
            bl_t = bl_t.unsqueeze(0)
        bl_t = torch.nan_to_num(bl_t, nan=0.0)
        target_full = bl_t + target
        target_full = _as_batched_hr(target_full)

        preds_all.append(ensemble.detach().cpu())
        truths_all.append(target_full.detach().cpu())
        mu_total_all.append(mu_total.detach().cpu())
        # Time can be int / numpy / Timestamp; we keep as-is.
        if 'time' in micro and micro['time'] is not None:
            tval = micro['time']
            # If sequence of times, take last
            try:
                if hasattr(tval, '__len__') and not isinstance(tval, str):
                    tval = tval[-1]
            except Exception:
                pass
            times_all.append(tval)
        else:
            times_all.append(None)
    n_done += 1
    if n_done % 10 == 0:
        elapsed = _time.time() - t0
        print(f'  [{n_done:3d}/{N_BATCHES_EVAL}] elapsed={elapsed:.1f}s  '
              f'preds={len(preds_all)}')

print()
print(f'[Cell 6] Done : {len(preds_all)} samples in {_time.time()-t0:.1f}s')
print(f'[Cell 6] ensemble shape (one) = {tuple(preds_all[0].shape)}')
print(f'[Cell 6] truth shape (one)    = {tuple(truths_all[0].shape)}')


In [ ]:
# === Cell 7 : Metriques deterministe + probabiliste ===
# Reuse les helpers definis dans scripts/eval_three_variants.py (importes inline).

def _pearson(a, b):
    a = a.flatten().float(); b = b.flatten().float()
    mask = torch.isfinite(a) & torch.isfinite(b)
    if mask.sum() < 2: return float('nan')
    a, b = a[mask], b[mask]
    a = a - a.mean(); b = b - b.mean()
    denom = (a.norm() * b.norm()).clamp(min=1e-12)
    return float((a * b).sum() / denom)

def _rmse(a, b):
    sq = (a - b).pow(2)
    return float(sq.nanmean().sqrt())

def _f1_at_percentile(pred, target, q):
    flat_t = target.flatten().float(); flat_p = pred.flatten().float()
    mask = torch.isfinite(flat_t) & torch.isfinite(flat_p)
    flat_t, flat_p = flat_t[mask], flat_p[mask]
    if flat_t.numel() < 100: return float('nan')
    thr = torch.quantile(flat_t, q).item()
    yt = (flat_t > thr).float(); yp = (flat_p > thr).float()
    tp = (yt * yp).sum().item()
    fp = ((1 - yt) * yp).sum().item()
    fn = (yt * (1 - yp)).sum().item()
    if tp + fp + fn == 0: return float('nan')
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    if precision + recall < 1e-12: return 0.0
    return 2 * precision * recall / (precision + recall)

def _crps_proxy(ensemble, target):
    """CRPS Hersbach estimator.

    ensemble : [N, K, 1, H, W]   N samples in time, K members
    target   : [N, 1, H, W]
    """
    if ensemble.dim() < 5 or ensemble.shape[1] < 2:
        return float('nan')
    # broadcast target to [N, 1, 1, H, W]
    tgt = target.unsqueeze(1)
    abs_diff_target = (ensemble - tgt).abs().nanmean(dim=1)            # [N,1,H,W]
    pairs = (ensemble.unsqueeze(1) - ensemble.unsqueeze(2)).abs()       # [N,K,K,1,H,W]
    abs_diff_pairs = pairs.nanmean(dim=(1, 2)) * 0.5                    # [N,1,H,W]
    crps = (abs_diff_target - abs_diff_pairs).nanmean()
    return float(crps)

# --- Stack ---
ens_stack   = torch.stack(preds_all,    dim=0)     # [N, K, 1, 1, H, W]
truth_stack = torch.stack(truths_all,   dim=0)     # [N, 1, 1, H, W]

# Squeeze the extra batch=1 dim coming from each item if present
# We expect ens_stack shape = [N, K, 1, 1, H, W] -> reduce to [N, K, 1, H, W]
if ens_stack.dim() == 6 and ens_stack.shape[2] == 1:
    ens_stack = ens_stack.squeeze(2)
if truth_stack.dim() == 5 and truth_stack.shape[1] == 1:
    truth_stack = truth_stack.squeeze(1)

print(f'[Cell 7] ens_stack shape   = {tuple(ens_stack.shape)}   (expect [N,K,1,H,W])')
print(f'[Cell 7] truth_stack shape = {tuple(truth_stack.shape)} (expect [N,1,H,W])')

# Ensemble mean
ens_mean = ens_stack.nanmean(dim=1)                # [N, 1, H, W]

rmse_final        = _rmse(ens_mean, truth_stack)
pearson           = _pearson(ens_mean, truth_stack)
f1_p95            = _f1_at_percentile(ens_mean, truth_stack, 0.95)
f1_p99            = _f1_at_percentile(ens_mean, truth_stack, 0.99)
crps_log1p        = _crps_proxy(ens_stack, truth_stack)
spread            = float(ens_stack.std(dim=1).nanmean())
spread_rmse_ratio = float(spread / max(rmse_final, 1e-12))

print()
print('=' * 60)
print('DETERMINISTIC + PROBABILISTIC METRICS (log1p space)')
print('=' * 60)
print(f'  RMSE_final          : {rmse_final:.5f}')
print(f'  Pearson             : {pearson:.5f}')
print(f'  F1 @ p95            : {f1_p95:.5f}')
print(f'  F1 @ p99            : {f1_p99:.5f}')
print(f'  CRPS (log1p)        : {crps_log1p:.5f}')
print(f'  Spread (ens std)    : {spread:.5f}')
print(f'  Spread / RMSE ratio : {spread_rmse_ratio:.5f}')


In [ ]:
# === Cell 8 : Climate indices (CDD/Rx1day/R10day) + PSD via aligned_eval ===
import numpy as _np
from st_cdgm.evaluation.aligned_eval import run_aligned_eval

# Build (T, H, W) arrays from ens_mean and truth_stack.
# Shapes : ens_mean [N,1,H,W], truth_stack [N,1,H,W]
pred_THW  = ens_mean.squeeze(1).numpy()    # [N, H, W]
truth_THW = truth_stack.squeeze(1).numpy() # [N, H, W]

# Times - aligned_eval needs len(times) == T
# If some times are None or non-datetime, fall back to a synthetic daily range.
def _coerce_times(tlist, n):
    """Return numpy datetime64 array of length n, or None if impossible."""
    valid = [t for t in tlist if t is not None]
    if len(valid) == n:
        try:
            return _np.array([_np.datetime64(t) for t in valid])
        except Exception:
            pass
    # Fall back: synthetic daily times starting 2010-01-01 (val range start)
    print(f'[Cell 8] times list incomplete ({len(valid)}/{n}); using synthetic daily range starting 2010-01-01')
    return _np.array([_np.datetime64('2010-01-01') + _np.timedelta64(i, 'D') for i in range(n)])

times_arr = _coerce_times(times_all, pred_THW.shape[0])

aligned_out_path = OUT_DIR / 'phase6_aligned_metrics.json'

cdd_bias      = float('nan')
rx1day_bias   = float('nan')
r10day_bias   = float('nan')
psd_distance  = float('nan')
indices_dict  = {}

try:
    aligned = run_aligned_eval(
        pred_fields=pred_THW,
        truth_fields=truth_THW,
        times=times_arr,
        out_path=str(aligned_out_path),
        gcm='ACCESS-CM2',
        run_variant='phase6_dualpath',
        in_distribution=True,
        space='log1p',
        thresh=1.0,
        k_samples=K_SAMPLES,
        psd_nx=int(H_HR),
        psd_ny=int(W_HR),
    )
    indices_dict = aligned.get('indices', {}) or {}
    psd_distance = float(aligned.get('psd_distance', float('nan')))

    # Indices keys vary; we try common naming used by cgan_aligned_metrics
    def _safe_bias(d, *keys):
        for k in keys:
            if k in d:
                try:
                    return float(d[k])
                except Exception:
                    pass
        return float('nan')

    cdd_bias    = _safe_bias(indices_dict, 'cdd_bias', 'CDD_bias', 'cdd')
    rx1day_bias = _safe_bias(indices_dict, 'rx1day_bias', 'Rx1day_bias', 'rx1day')
    r10day_bias = _safe_bias(indices_dict, 'r10day_bias', 'R10day_bias', 'r10day', 'rx10day_bias')

    print()
    print('=' * 60)
    print('CLIMATE INDICES + PSD (via aligned_eval)')
    print('=' * 60)
    print(f'  CDD bias       : {cdd_bias:.5f}')
    print(f'  Rx1day bias    : {rx1day_bias:.5f}')
    print(f'  R10day bias    : {r10day_bias:.5f}')
    print(f'  PSD distance   : {psd_distance:.5f}')
    print(f'  Aligned JSON   : {aligned_out_path}')
    print()
    print('  Full indices keys :', list(indices_dict.keys())[:20])
except Exception as e:
    print(f'[Cell 8] aligned_eval FAILED : {type(e).__name__}: {e}')
    import traceback as _tb
    _tb.print_exc()


In [ ]:
# === Cell 9 : Sauvegarder JSON + table comparative + verdict ===
results = {
    'phase':             'phase6_dualpath_evaluation',
    'ckpt_dualpath':     str(CKPT_DUALPATH),
    'ckpt_stage2':       str(CKPT_STAGE2),
    'sigma_data':        float(SIGMA_DATA_NEW),
    'n_batches':         int(N_BATCHES_EVAL),
    'n_samples':         int(ens_stack.shape[0]),
    'k_samples':         int(K_SAMPLES),
    'n_steps':           int(N_STEPS),
    'rmse_final':        float(rmse_final),
    'pearson':           float(pearson),
    'f1_p95':            float(f1_p95),
    'f1_p99':            float(f1_p99),
    'crps_log1p':        float(crps_log1p),
    'spread':            float(spread),
    'spread_rmse_ratio': float(spread_rmse_ratio),
    'psd_distance':      float(psd_distance),
    'cdd_bias':          float(cdd_bias),
    'rx1day_bias':       float(rx1day_bias),
    'r10day_bias':       float(r10day_bias),
    'indices_raw':       indices_dict,
}

results_path = OUT_DIR / 'phase6_eval_results.json'
results_path.write_text(json.dumps(results, indent=2, default=str), encoding='utf-8')
print(f'[Cell 9] Results saved to : {results_path}')

# --- Comparison table vs Phase 5 (Oracle) and CorrDiff baseline ---
print()
print('=' * 72)
print('COMPARAISON Phase 6 DualPath vs Phase 5 Oracle vs CorrDiff baseline')
print('=' * 72)
print(f'  {"Metric":<22}{"Phase 6":>14}{"Phase 5 Oracle":>18}{"CorrDiff":>14}')
print('  ' + '-' * 68)
_p5_rmse, _p5_pear, _p5_f1   = 0.141, 0.767, 0.453
_nc_rmse, _nc_pear, _nc_f1   = 0.124, 0.834, 0.550
print(f'  {"RMSE_final":<22}{rmse_final:>14.4f}{_p5_rmse:>18.4f}{_nc_rmse:>14.4f}')
print(f'  {"Pearson":<22}{pearson:>14.4f}{_p5_pear:>18.4f}{_nc_pear:>14.4f}')
print(f'  {"F1 @ p99":<22}{f1_p99:>14.4f}{_p5_f1:>18.4f}{_nc_f1:>14.4f}')
print(f'  {"F1 @ p95":<22}{f1_p95:>14.4f}{"-":>18}{"-":>14}')
print(f'  {"CRPS (log1p)":<22}{crps_log1p:>14.4f}{"-":>18}{"-":>14}')
print(f'  {"Spread / RMSE":<22}{spread_rmse_ratio:>14.4f}{"-":>18}{"-":>14}')
print(f'  {"PSD distance":<22}{psd_distance:>14.4f}{"-":>18}{"-":>14}')

# --- Verdict ---
print()
print('=' * 72)
print('VERDICT Phase 6 (vs Phase 5 Oracle baseline)')
print('=' * 72)
_d_rmse   = rmse_final - _p5_rmse
_d_pear   = pearson - _p5_pear
_d_f1p99  = f1_p99 - _p5_f1
print(f'  Delta RMSE     : {_d_rmse:+.4f}  ({"BETTER" if _d_rmse < 0 else "WORSE"})')
print(f'  Delta Pearson  : {_d_pear:+.4f}  ({"BETTER" if _d_pear > 0 else "WORSE"})')
print(f'  Delta F1 @ p99 : {_d_f1p99:+.4f}  ({"BETTER" if _d_f1p99 > 0 else "WORSE"})')
print()
print(f'[Cell 9] DONE. JSON => {results_path}')
